## Imports

In [22]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

from multiprocessing import Pool
from functools import partial
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## For ETTh1

In [33]:
from data_loader import Dataset_ETT_hour

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT-small"
data_id = "ETTh1"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_ETT_hour(
        root_path=data_root,
        data_path=f"{data_id}.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(8449, 96, 7) (8449, 96, 7) (8449, 96, 4)
(96, 96)
(96, 96)
(96, 96)
(8353, 96, 7) (8353, 192, 7) (8353, 96, 4)
(96, 96)
(192, 192)
(96, 96)
(8209, 96, 7) (8209, 336, 7) (8209, 96, 4)
(96, 96)
(336, 336)
(96, 96)
(7825, 96, 7) (7825, 720, 7) (7825, 96, 4)
(96, 96)
(720, 720)
(96, 96)


## For ETTh2

In [34]:
from data_loader import Dataset_ETT_hour

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT-small"
data_id = "ETTh2"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_ETT_hour(
        root_path=data_root,
        data_path=f"{data_id}.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(8449, 96, 7) (8449, 96, 7) (8449, 96, 4)
(96, 96)
(96, 96)
(96, 96)
(8353, 96, 7) (8353, 192, 7) (8353, 96, 4)
(96, 96)
(192, 192)
(96, 96)
(8209, 96, 7) (8209, 336, 7) (8209, 96, 4)
(96, 96)
(336, 336)
(96, 96)
(7825, 96, 7) (7825, 720, 7) (7825, 96, 4)
(96, 96)
(720, 720)
(96, 96)


## For ETTm1

In [35]:
from data_loader import Dataset_ETT_minute

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT-small"
data_id = "ETTm1"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_ETT_minute(
        root_path=data_root,
        data_path=f"{data_id}.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(34369, 96, 7) (34369, 96, 7) (34369, 96, 4)
(96, 96)
(96, 96)
(96, 96)
(34273, 96, 7) (34273, 192, 7) (34273, 96, 4)
(96, 96)
(192, 192)
(96, 96)
(34129, 96, 7) (34129, 336, 7) (34129, 96, 4)
(96, 96)
(336, 336)
(96, 96)
(33745, 96, 7) (33745, 720, 7) (33745, 96, 4)
(96, 96)
(720, 720)
(96, 96)


## For ETTm2

In [36]:
from data_loader import Dataset_ETT_minute

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT-small"
data_id = "ETTm2"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_ETT_minute(
        root_path=data_root,
        data_path=f"{data_id}.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(34369, 96, 7) (34369, 96, 7) (34369, 96, 4)
(96, 96)
(96, 96)
(96, 96)
(34273, 96, 7) (34273, 192, 7) (34273, 96, 4)
(96, 96)
(192, 192)
(96, 96)
(34129, 96, 7) (34129, 336, 7) (34129, 96, 4)
(96, 96)
(336, 336)
(96, 96)
(33745, 96, 7) (33745, 720, 7) (33745, 96, 4)
(96, 96)
(720, 720)
(96, 96)


## For Weather

In [37]:
from data_loader import Dataset_Custom

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/weather"
data_id = "Weather"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_Custom(
        root_path=data_root,
        data_path=f"weather.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


Head lines of raw dataframe:
                  date  p (mbar)  T (degC)  Tpot (K)  Tdew (degC)  rh (%)  \
0  2020-01-01 00:10:00   1008.89      0.71    273.18        -1.33    86.1   
1  2020-01-01 00:20:00   1008.76      0.75    273.22        -1.44    85.2   
2  2020-01-01 00:30:00   1008.66      0.73    273.21        -1.48    85.1   
3  2020-01-01 00:40:00   1008.64      0.37    272.86        -1.64    86.3   
4  2020-01-01 00:50:00   1008.61      0.33    272.82        -1.50    87.4   

   VPmax (mbar)  VPact (mbar)  VPdef (mbar)  sh (g/kg)  H2OC (mmol/mol)  \
0          6.43          5.54          0.89       3.42             5.49   
1          6.45          5.49          0.95       3.39             5.45   
2          6.44          5.48          0.96       3.39             5.43   
3          6.27          5.41          0.86       3.35             5.37   
4          6.26          5.47          0.79       3.38             5.42   

   rho (g/m**3)  wv (m/s)  max. wv (m/s)  wd (deg)  rain 

## For Traffic

In [38]:
from data_loader import Dataset_Custom

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/traffic"
data_id = "Traffic"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_Custom(
        root_path=data_root,
        data_path=f"traffic.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


Head lines of raw dataframe:
                  date       0       1       2       3       4       5  \
0  2016-07-01 02:00:00  0.0048  0.0146  0.0289  0.0142  0.0064  0.0232   
1  2016-07-01 03:00:00  0.0072  0.0148  0.0350  0.0174  0.0084  0.0240   
2  2016-07-01 04:00:00  0.0040  0.0101  0.0267  0.0124  0.0049  0.0170   
3  2016-07-01 05:00:00  0.0039  0.0060  0.0218  0.0090  0.0029  0.0118   
4  2016-07-01 06:00:00  0.0042  0.0055  0.0191  0.0082  0.0024  0.0095   

        6       7       8       9      10      11      12      13      14  \
0  0.0162  0.0242  0.0341  0.0375  0.0144  0.0098  0.0157  0.0216  0.0345   
1  0.0201  0.0338  0.0434  0.0381  0.0162  0.0114  0.0192  0.0239  0.0392   
2  0.0127  0.0255  0.0332  0.0309  0.0122  0.0074  0.0137  0.0153  0.0233   
3  0.0088  0.0163  0.0211  0.0199  0.0077  0.0060  0.0098  0.0082  0.0142   
4  0.0064  0.0087  0.0144  0.0226  0.0055  0.0053  0.0088  0.0036  0.0101   

       15      16      17      18      19      20      21      

## For ECL

In [39]:
from data_loader import Dataset_Custom

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/electricity"
data_id = "ECL"


for pred_len in [96, 192, 336, 720]:

    data_set = Dataset_Custom(
        root_path=data_root,
        data_path=f"electricity.csv",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


Head lines of raw dataframe:
                  date     0     1      2      3      4       5     6       7  \
0  2016-07-01 02:00:00  14.0  69.0  234.0  415.0  215.0  1056.0  29.0   840.0   
1  2016-07-01 03:00:00  18.0  92.0  312.0  556.0  292.0  1363.0  29.0  1102.0   
2  2016-07-01 04:00:00  21.0  96.0  312.0  560.0  272.0  1240.0  29.0  1025.0   
3  2016-07-01 05:00:00  20.0  92.0  312.0  443.0  213.0   845.0  24.0   833.0   
4  2016-07-01 06:00:00  22.0  91.0  312.0  346.0  190.0   647.0  16.0   733.0   

       8      9     10     11     12     13     14      15    16     17  \
0  226.0  265.0  179.0  148.0  112.0  171.0  229.0  1001.0  49.0  162.0   
1  271.0  340.0  235.0  192.0  143.0  213.0  301.0  1223.0  64.0  216.0   
2  270.0  300.0  221.0  171.0  132.0  185.0  261.0  1172.0  61.0  197.0   
3  179.0  211.0  170.0  149.0  116.0  151.0  209.0   813.0  40.0  173.0   
4  186.0  179.0  142.0  170.0   99.0  136.0  148.0   688.0  29.0  144.0   

      18     19    20      21    

## For PEMS03

In [41]:
from data_loader import Dataset_PEMS

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/PEMS"
data_id = "PEMS03"


for pred_len in [12, 24, 36, 48]:

    data_set = Dataset_PEMS(
        root_path=data_root,
        data_path=f"PEMS03.npz",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        if tag == "input_mark":
            continue

        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(15617, 96, 358) (15617, 12, 358) (15617, 96, 1)
(96, 96)
(12, 12)
(15605, 96, 358) (15605, 24, 358) (15605, 96, 1)
(96, 96)
(24, 24)
(15593, 96, 358) (15593, 36, 358) (15593, 96, 1)
(96, 96)
(36, 36)
(15581, 96, 358) (15581, 48, 358) (15581, 96, 1)
(96, 96)
(48, 48)


## For PEMS08

In [42]:
from data_loader import Dataset_PEMS

seq_len = 96
data_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/PEMS"
data_id = "PEMS08"


for pred_len in [12, 24, 36, 48]:

    data_set = Dataset_PEMS(
        root_path=data_root,
        data_path=f"PEMS08.npz",
        flag="train",
        size=[seq_len, seq_len // 2, pred_len],
        features="M",
        target="OT",
        timeenc=1,
        freq='h'
    )

    input_seq, label_seq, input_mark_seq = [], [], []
    for i in range(len(data_set)):
        inp, label, inp_mark, _ = data_set[i]
        label = label[-pred_len:]
        input_seq.append(inp)
        label_seq.append(label)
        input_mark_seq.append(inp_mark)

    input_seq = np.array(input_seq)        # shape: [N, S, D]
    label_seq = np.array(label_seq)        # shape: [N, P, D]
    input_mark_seq = np.array(input_mark_seq)  # shape: [N, S, mark_dim]

    print(input_seq.shape, label_seq.shape, input_mark_seq.shape)

    for seq, tag, length in zip([input_seq, label_seq, input_mark_seq], ["input", "output", "input_mark"], [seq_len, pred_len, seq_len]):
        if tag == "input_mark":
            continue

        Sigma_list = []
        for d in range(seq.shape[-1]):
            seqT = seq[..., d].T
            cov_matrix = np.cov(seqT)
            diag_vec = np.diag(cov_matrix)

            if (diag_vec < 1e-4).any():
                continue

            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1
            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)

        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
        q_mat = np.flip(eigenvectors.T, axis=0)
        print(q_mat.shape)

        proj_dir = f'/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/projections/EVD/{data_id}/sp0/{tag}/{length}'
        os.makedirs(proj_dir, exist_ok=True)
        np.save(os.path.join(proj_dir, "eigenvectors.npy"), q_mat)


(10606, 96, 170) (10606, 12, 170) (10606, 96, 1)
(96, 96)
(12, 12)
(10594, 96, 170) (10594, 24, 170) (10594, 96, 1)
(96, 96)
(24, 24)
(10582, 96, 170) (10582, 36, 170) (10582, 96, 1)
(96, 96)
(36, 36)
(10570, 96, 170) (10570, 48, 170) (10570, 96, 1)
(96, 96)
(48, 48)
